# 05 — Most Diversified Portfolio

The Most Diversified Portfolio (MDP) maximises the diversification ratio
$DR(w) = w'\sigma / \sqrt{w'\Sigma w}$ over long-only, fully-invested
weights — the ratio of the weighted-average volatility an investor would
bear if the assets were uncorrelated to the actual realised portfolio
volatility. Unlike GMV, which rewards both low volatility and low
correlation, MDP rewards decorrelation alone.

Notebook 04 found a persistent UUP corner in GMV and MaxSharpe that survives
covariance shrinkage — not sample-covariance noise, since it doesn't move
much when the covariance estimator changes. This notebook asks whether a
risk-based method that scores decorrelation on its own, rather than as a
side effect of minimising variance, removes that corner.

## §1 Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import maplab as ml
import _nb05_helpers as h
from maplab import (
    Panel, apply_style, GMV, MaxSharpe, EqualWeight, MostDiversified,
    backtest, summary, mdp_closed_form,
)
from maplab.models import _min_variance_long_only, _mdp_long_only

apply_style()
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

prices = ml.load_prices()
log_returns = ml.to_log_returns(prices)        # for estimation (mu, Sigma)
simple_returns = ml.to_simple_returns(prices)  # for compounding the backtest
rf_daily = ml.load_rf_returns()["BIL"]
panel = Panel({"prices": prices, "returns": log_returns, "rf": ml.load_rf_returns()})
split_ts = pd.Timestamp(ml.TRAIN_TEST_SPLIT)

print(f"prices: {prices.shape[0]:,} days x {prices.shape[1]} assets "
      f"({prices.index.min().date()} -> {prices.index.max().date()})")
print(f"train/test split: {split_ts.date()}")

rdates = ml.rebalance_dates(log_returns.index)
rdates = rdates[rdates >= ml.first_eligible_rebalance(log_returns.index)]
assert len(rdates) == 208, f"expected 208 rebalance labels, got {len(rdates)}"
print(f"rebalance labels: n={len(rdates)}  first={rdates[0].date()}  last={rdates[-1].date()}")

In [ ]:
ml.inference.render_registration(ml.inference.load_registration("nb05"))

## §3 Theory

- **Long-only max DR = tangency QP with excess := σ.** Maximising
  $DR(w)=w'\sigma/\sqrt{w'\Sigma w}$ over $\{w:\mathbf 1'w=1,\ w\ge 0\}$ is
  solved by the same convex reformulation as `MaxSharpe`: minimise
  $y'\Sigma y$ s.t. $\sigma'y=1$, $y\ge 0$; $w=y/\mathbf 1'y$. This is exact
  (not an approximation) because $\sigma>0$ always, so the reformulation is
  never degenerate — `_mdp_long_only` is a one-line call into
  `_tangency_long_only` with `excess := sigma`.
- **Identity.** The risk weights $x=w\circ\sigma/(w'\sigma)$ of the MDP
  solution equal the long-only GMV solution on the correlation matrix
  $C=D^{-1}\Sigma D^{-1}$. GMV on $\Sigma$ prefers low volatility *and* low
  correlation; MDP keeps only the correlation preference.
- **Special cases.** Constant correlation ⇒ MDP is inverse-vol. Equal vols
  ⇒ MDP is GMV.
- **Scale invariance.** MDP's risk weights are unchanged if asset $i$ is
  rescaled by any $c_i>0$ (only $C$ matters for the risk weights).
  `sample_cov` is equivariant under this rescaling; the Ledoit-Wolf target
  $mI$ is not (shrinking toward a common scalar variance mixes vols across
  assets) — so MDP(LW) differs from MDP(S) partly for a reason unrelated to
  estimation noise.

In [ ]:
asof_2022 = pd.Timestamp("2022-12-31")
_, Sigma_2022 = GMV(cov_estimator=ml.sample_cov)._estimate(panel, asof_2022)
w_mdp_unc_2022 = mdp_closed_form(Sigma_2022)
gross_2022 = float(w_mdp_unc_2022.abs().sum())
print(f"unconstrained MDP gross leverage sum|w| at 2022-12-31 = {gross_2022:.4f}")
print("(why long-only: the unconstrained MDP solution shorts/levers rather than staying fully invested)")

## §4 Backtests

`IV` is defined locally below — a notebook-local `Strategy` subclass,
inverse-vol weights off the same 252-day sample covariance window GMV/MDP
use (never backtested with a shrinkage estimator). Six fresh instances,
same cost/rebalance/warm-up harness as notebooks 01–04.

In [ ]:
strategies = {
    "GMV(S)": GMV(cov_estimator=ml.sample_cov),
    "MaxSharpe(S)": MaxSharpe(cov_estimator=ml.sample_cov),
    "EW": EqualWeight(),
    "MDP(S)": MostDiversified(cov_estimator=ml.sample_cov),
    "MDP(LW)": MostDiversified(cov_estimator=ml.ledoit_wolf_cov),
    "IV": ml.InverseVol(),
}

results = {}
for name, strat in strategies.items():
    net, wlog, diag = backtest(strat, simple_returns, panel)
    results[name] = {"net": net, "wlog": wlog, "diag": diag, "strat": strat}

print(f"{len(results)} runs completed.")

### Reproduction gate

GMV(S), MaxSharpe(S), and EW must match notebooks 01/03/04 in full/train/test
Sharpe to 2 dp: GMV 0.62/0.59/0.76, MaxSharpe 0.72/0.60/1.13, EW 0.74/0.70/0.91.
If this doesn't hold the harness or estimator wiring changed something and
everything downstream is suspect — so this stops the notebook (`assert`)
rather than just printing a warning.

In [ ]:
rows = []
for name, r in results.items():
    net = r["net"]
    diag = r["diag"]
    turnover = diag["turnover_per_rebalance"]
    win_defs = {
        "full": (net.index >= net.index.min(), turnover.index >= turnover.index.min()),
        "train (<= split)": (net.index <= split_ts, turnover.index <= split_ts),
        "test (> split)": (net.index > split_ts, turnover.index > split_ts),
    }
    for window_name, (net_sel, t_sel) in win_defs.items():
        t_window = turnover.loc[t_sel]
        s = summary(net.loc[net_sel], t_window, rf=rf_daily)
        s["strategy"] = name
        s["window"] = window_name
        rows.append(s)

summary_table = pd.DataFrame(rows).set_index(["strategy", "window"])
summary_table = summary_table[["ann_return", "ann_vol", "sharpe", "max_dd", "calmar", "hit_rate", "ann_turnover"]]
summary_table["cost_drag"] = summary_table["ann_turnover"] * ml.COST_BPS / 1e4
summary_table.round(4)

In [ ]:
ml.inference.check_reproduction(summary_table, {"GMV(S)": "GMV_sample", "MaxSharpe(S)": "MaxSharpe_sample", "EW": "EW"})

print("REPRODUCTION GATE: PASSED -- GMV(S), MaxSharpe(S), EW match notebooks 01/03/04 to 2 dp.")

## §6 Mechanism checks

Over all 208 rebalances, for both the sample and Ledoit-Wolf estimators:
the risk-weight identity ($x_{MDP}$ vs long-only GMV on $C$) and DR
optimality of the MDP solution against GMV/EW/IV on the same $\Sigma$,
recomputed fresh (not read off the §4 backtests) so this is an independent
check of the solver, not a tautology. `retry_dates` bookkeeping is read
directly off the §4 strategy instances.

In [ ]:
mech_table = h.mechanism_checks(panel, rdates)
print(mech_table)

for _, row in mech_table.iterrows():
    assert row["max_identity_diff"] <= 1e-5, f"identity check failed: {row['max_identity_diff']}"
    assert row["min_dr_margin"] >= -1e-10, f"DR optimality failed: {row['min_dr_margin']}"

print("Mechanism checks: PASSED (identity <= 1e-5, DR optimality >= -1e-10) at all 208 rebalances, both estimators.")

n_retry_s = len(results["MDP(S)"]["strat"].retry_dates)
n_retry_lw = len(results["MDP(LW)"]["strat"].retry_dates)
print(f"retry_dates: MDP(S)={n_retry_s}  MDP(LW)={n_retry_lw}")

## §7 Snapshot: 2014-12-31 / 2017-12-31 / 2022-12-31 (sample)

Weights (GMV/MDP/IV/EW) and metrics off the same read-only snapshot dates
used to pre-register H1/H2, plus the UUP decomposition: g = UUP weight in
GMV, r = UUP risk share in MDP (= UUP weight in the long-only GMV on the
correlation matrix), m = UUP weight in MDP.

In [ ]:
DATES = [pd.Timestamp(d) for d in ["2014-12-31", "2017-12-31", "2022-12-31"]]

weight_tables, snapshot_metrics, uup_table = h.snapshot_tables(panel, results, DATES)

for d, wt in weight_tables.items():
    print(f"\n{d}  weights:")
    print(wt)

print("\nmetrics:")
print(snapshot_metrics.round(4))

print("\nUUP decomposition (g=GMV weight, r=MDP risk share, m=MDP weight):")
print(uup_table.round(4))

## §8 Summary: full / train / test

Ret, Vol, Sharpe (+ SE), MaxDD, TO/yr for all six runs.

In [ ]:
summary_rows2 = []
for name, r in results.items():
    net_full = r["net"]
    for window_name, sel in [
        ("full", net_full.index >= net_full.index.min()),
        ("train (<= split)", net_full.index <= split_ts),
        ("test (> split)", net_full.index > split_ts),
    ]:
        net_win = net_full.loc[sel]
        sr = float(summary_table.loc[(name, window_name), "sharpe"])
        T_years = len(net_win) / ml.TRADING_DAYS
        se = ml.inference.sharpe_se(sr, T_years)
        summary_rows2.append({
            "strategy": name, "window": window_name,
            "ann_return": float(summary_table.loc[(name, window_name), "ann_return"]),
            "ann_vol": float(summary_table.loc[(name, window_name), "ann_vol"]),
            "sharpe": sr, "sharpe_se": se,
            "max_dd": float(summary_table.loc[(name, window_name), "max_dd"]),
            "ann_turnover": float(summary_table.loc[(name, window_name), "ann_turnover"]),
        })
summary_table2 = pd.DataFrame(summary_rows2).set_index(["strategy", "window"])
summary_table2.round(4)

## §9 H1 — effective N

In [ ]:
effN_gmv_ts = 1.0 / (results["GMV(S)"]["wlog"] ** 2).sum(axis=1)
effN_mdp_ts = 1.0 / (results["MDP(S)"]["wlog"] ** 2).sum(axis=1)

fig = ml.plotting.concentration_panel(
    results, ["MDP(S)", "GMV(S)"], rows=("effN",), ref=None, split_ts=split_ts, figsize=(9.5, 4.5),
    titles=["H1: effective N, MDP(S) vs GMV(S)"], split_label="train/test split", xlabel=None,
)
plt.show()

In [ ]:
le_full = int((effN_mdp_ts <= effN_gmv_ts).sum())
n_full = len(effN_mdp_ts)
test_mask = effN_mdp_ts.index > split_ts
le_test = int((effN_mdp_ts.loc[test_mask] <= effN_gmv_ts.loc[test_mask]).sum())
n_test = int(test_mask.sum())
median_mdp = float(effN_mdp_ts.median())
median_gmv = float(effN_gmv_ts.median())

print(f"H1: {le_full}/{n_full} ({100*le_full/n_full:.1f}%), test {le_test}/{n_test}")
print(f"median effN: MDP(S)={median_mdp:.4f}  GMV(S)={median_gmv:.4f}")

## §10 H2 — UUP weight

In [ ]:
w_uup_gmv_ts = results["GMV(S)"]["wlog"]["UUP"]
w_uup_mdp_ts = results["MDP(S)"]["wlog"]["UUP"]

fig = ml.plotting.concentration_panel(
    results, ["MDP(S)", "GMV(S)"], rows=("highlight",), ref=None, split_ts=split_ts, figsize=(9.5, 4.5),
    titles=["H2: UUP weight, MDP(S) vs GMV(S)"], split_label="train/test split", xlabel=None,
)
plt.show()

In [ ]:
ge_full = int((w_uup_mdp_ts >= w_uup_gmv_ts).sum())
n_full2 = len(w_uup_mdp_ts)
test_mask2 = w_uup_mdp_ts.index > split_ts
ge_test = int((w_uup_mdp_ts.loc[test_mask2] >= w_uup_gmv_ts.loc[test_mask2]).sum())
n_test2 = int(test_mask2.sum())

uup_2022_gmv = float(w_uup_gmv_ts.loc[pd.Timestamp("2022-12-31")])
uup_2022_mdp = float(w_uup_mdp_ts.loc[pd.Timestamp("2022-12-31")])

print(f"H2: {ge_full}/{n_full2} ({100*ge_full/n_full2:.1f}%), test {ge_test}/{n_test2}")
print(f"2022-12-31: GMV(S) UUP={uup_2022_gmv:.4f}  MDP(S) UUP={uup_2022_mdp:.4f}")

## §11 Paired tests, β decomposition, cost drag

Pairs: MDP(S) − GMV(S), MDP(S) − EW, MDP(LW) − MDP(S), MDP(S) − IV. Plain
OLS (no HAC — same caveat as notebooks 03/04), realized daily simple excess
return on the market's simple excess return; the identity check is 0 by OLS
construction (mean residual is exactly 0 for an OLS fit with an intercept),
not a residual to be interpreted.

In [ ]:
pairs = [
    ("MDP(S)", "GMV(S)"),
    ("MDP(S)", "EW"),
    ("MDP(LW)", "MDP(S)"),
    ("MDP(S)", "IV"),
]

paired_table, paired_series = ml.inference.paired_table(results, pairs, split_ts, summary_table)
print(paired_table.round(4))

print("\ncost drag %/yr per run (full window):")
for name in results:
    print(f"  {name:12s} {float(summary_table.loc[(name, 'full'), 'cost_drag']) * 100:.4f}%")

In [ ]:
MKT_simple = (simple_returns["SPY"] - rf_daily).dropna()

capm_table = ml.inference.capm_table(results, ["MDP(S)", "GMV(S)", "EW", "MDP(LW)", "IV"],
                                     MKT_simple, rf_daily)
print(capm_table.round(4))

In [ ]:
# Direct regression of each pair's diff series on MKT: alpha_diff has its
# own t-stat (unlike alpha_a - alpha_b, which ignores the covariance
# between the two separate alpha estimates) -- this is what H4/H5's
# "Delta-alpha |t| < 2" conditions are evaluated against.
diff_capm_table = ml.inference.diff_regression(paired_series, pairs, MKT_simple)
print(diff_capm_table.round(4))

In [ ]:
decomp = ml.inference.beta_decomposition(capm_table, paired_table, diff_capm_table, pairs,
                                         results, rf_daily, MKT_simple)
print(ml.inference.format_beta_decomposition(decomp, capm_table, diff_capm_table), end="")

## §12 Sensitivity: LW and IV distance

In [ ]:
wlog_mdp_s = results["MDP(S)"]["wlog"]
wlog_mdp_lw = results["MDP(LW)"]["wlog"]
wlog_iv = results["IV"]["wlog"]

halfL1_lw_vs_s = 0.5 * (wlog_mdp_lw - wlog_mdp_s).abs().sum(axis=1)
halfL1_mdp_vs_iv = 0.5 * (wlog_mdp_s - wlog_iv).abs().sum(axis=1)
effN_mdp_lw_ts = 1.0 / (wlog_mdp_lw ** 2).sum(axis=1)

print(f"half-L1(MDP(LW), MDP(S)): median={float(halfL1_lw_vs_s.median()) * 100:.4f}%  max={float(halfL1_lw_vs_s.max()) * 100:.4f}%")
print(f"half-L1(MDP(S), IV):      median={float(halfL1_mdp_vs_iv.median()) * 100:.4f}%  max={float(halfL1_mdp_vs_iv.max()) * 100:.4f}%")
print(f"effN(MDP(LW)): median={float(effN_mdp_lw_ts.median()):.4f}")

## §13 TAKEAWAY

- **H1 — supported:** effN(MDP) ≤ effN(GMV) on 146/208 (70.2%) rebalances
  (test 34/40, 85.0%); median effN MDP(S) = 3.35 vs GMV(S) = 3.79.
- **H2 — supported:** UUP weight MDP ≥ GMV on 187/208 (89.9%) rebalances
  (test 38/40, 95.0%); the corner persists at 2022-12-31 (GMV 58.12% vs
  MDP 59.22%).
- **H3 — null holds, narrowly:** MDP(S) − GMV(S) +0.64%/yr full (t = 1.98)
  (just below the \|t\| ≥ 2 threshold), +1.10%/yr test (t = 1.68), both
  \|t\| < 2. Δα = +0.35%/yr (direct diff-regression, t = 1.12) vs Δβ part =
  +0.29%/yr; identity check −0.00% (0 by OLS construction).
- **H4 — supported, borderline:** MDP(S) − EW −3.17%/yr full (t = −2.00),
  test mean also negative (−3.20%/yr, though the test window's own t is
  only −0.99). \|Δβ part\| = 4.19% > \|Δα\| = 1.02%, and Δα's own t (direct
  diff-regression) = 1.00 < 2 — consistent with a beta-driven shortfall
  rather than an alpha effect, as the hypothesis predicted. The full-window
  t sits exactly at the −2 threshold, so this reads as a marginal pass, not
  a decisive one.
- **H5 — null holds:** MDP(LW) − MDP(S) +0.20%/yr full (t = 1.31), +0.14%/yr
  test (t = 0.53), both \|t\| < 2. MDP(S) − IV Δα (direct diff-regression)
  +0.59%/yr (t = 0.71), \|t\| < 2.

**nb04 link:** at 2022-12-31 the UUP weight is 58.12% in GMV and 59.22% in
MDP, so MDP does not remove the corner that survived covariance shrinkage
in nb04; across all rebalances MDP holds at least as much UUP as GMV on
187/208 (test 38/40). MDP's risk share in UUP is 41.67% — the largest of
any asset — with UUP's mean pairwise correlation at −0.4659, i.e. the
long-only GMV on the correlation matrix (vol preference removed) still
concentrates in UUP. At that date UUP was also the lowest-vol asset (rank
1/13), so the 1/σ conversion from risk share to cash weight raises its
weight further. By the pre-set D2 rule (m ≥ g) this is correlation-driven:
a decorrelation-only objective keeps the corner.